# Train the brick proposal model (M12b step 2)

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When cell 2 asks, upload `brick_proposal.zip` (made by `python training/pack.py`).
3. Training runs about 50 minutes. At the end the browser downloads `proposal_weights.zip`.
4. Put its three files in `out/proposal/` of the repository.

If the runtime disconnects, set `USE_DRIVE = True` below. Training then checkpoints to Google Drive, and **Run all** again resumes it (upload the zip again).

In [ ]:
# Settings
D_MODEL = 256      # 384 for a bigger model (~11M params) if the first one underfits
MINUTES = 50       # training time limit; the best weights so far are exported when it is reached
STEPS = 8000
USE_DRIVE = False  # True: checkpoints in Google Drive (survives a disconnect; asks for permission)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Upload brick_proposal.zip and unpack it
import os, zipfile
from google.colab import files
os.makedirs('/content/brick', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
zipfile.ZipFile(name).extractall('/content/brick')
os.chdir('/content/brick')
OUT = 'out/proposal'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/brick_proposal'
print(os.listdir('data/tokens'), 'output:', OUT)

In [ ]:
# Train (resumes from OUT/checkpoint.pt if it exists)
!python training/train.py --data data/tokens --out {OUT} --d-model {D_MODEL} --steps {STEPS} --minutes {MINUTES}

In [ ]:
# The numpy inference used on the CPU must match PyTorch on the trained weights
!PYTHONPATH=src python training/parity.py check --weights {OUT}/proposal.npz --data data/tokens

In [ ]:
# Download the weights, vocabulary and training log
import zipfile
with zipfile.ZipFile('proposal_weights.zip', 'w') as z:
    for f in ('proposal.npz', 'vocab.json', 'log.jsonl'):
        z.write(os.path.join(OUT, f), f)
files.download('proposal_weights.zip')